## Data Processing – Replicating “I Will Survive: Predicting Business Failures from Customer Ratings”

The following pipeline mirrors the data preparation and analysis steps of the Marketing Science case study and stores them in a Pickle file for further analysis.


In [1]:
import sys
from pathlib import Path

# add projects root directory to the system path to enable importing custom modules (e.g., from the "helpers" folder).
sys.path.append(str(Path("..").resolve()))

# Imports
import pandas as pd
import numpy as np


SEED = 42  # random seed for reproducability
SET_ORIGINAL_INDICES = False  # if set to true, the original paper indices are selected

np.random.seed(SEED)

In [2]:
from constants import DATA_FOLDER

# Dataframes
reviews = pd.read_csv(DATA_FOLDER / "reviews.csv")
business_covariates = pd.read_csv(DATA_FOLDER / "business_covariates.csv")

In [3]:
business_covariates.columns

Index(['business_id', 'name', 'neighborhood', 'address', 'city', 'state',
       'postal_code', 'latitude', 'longitude', 'stars', 'review_count',
       'is_open', 'categories', 'Checkin', 'chain', 'density', 'TRAIN',
       'category', 'FT', 'Price.Level', 'Restaurant.Size', 'Number.of.Seats',
       'ZRI', 'Distance.To.City.Centre'],
      dtype='object')

In [4]:
# create indices for training evaluation and calibration

from constants import CALIBRATION_INDICES, EVAL_INDICES, TRAIN_INDICES

# get original indices
if SET_ORIGINAL_INDICES:
    train_indices = TRAIN_INDICES
    calibration_indices = CALIBRATION_INDICES
    eval_indices = EVAL_INDICES

# get indices based on random seed
else:
    indices = np.random.permutation(len(business_covariates))
    indices_val_cal = np.random.permutation(
        np.arange(500, len(business_covariates))
    )  # range 500-921 (because of sorting)

    train_indices = indices[:500]  # take 500 random samples
    calibration_indices = indices_val_cal[:100]  # take 100 random out of range 500-921
    eval_indices = indices_val_cal[100:]  # take 321 random out range 500-921

In [5]:
assert (business_covariates.get("TRAIN")).sum() == 0, "training set already assigned!"

# set 'TRAIN' variable to 1 for train_indices, 0 otherwise
business_covariates.loc[train_indices, "TRAIN"] = 1

# sort business_covariates so that rows with Train==1 come first
business_covariates = business_covariates.sort_values(
    by="TRAIN", ascending=False
).reset_index(
    drop=True
)  # it is possible to retreive all training data with :500

n_train = len(train_indices)  # number of training samples (500)

print(f"Train/Calibration/Eval indices created:")
print(f"  Train: {len(train_indices)} samples")
print(f"  Calibration: {len(calibration_indices)} samples")
print(f"  Eval: {len(eval_indices)} samples")

Train/Calibration/Eval indices created:
  Train: 500 samples
  Calibration: 100 samples
  Eval: 321 samples


In [6]:
# initialize list with data needed for stan

ratings = []
sentiment = []
days = []
time = []
age = []

business_ids = business_covariates["business_id"].values

# convert date string into datetime object
reviews["date"] = pd.to_datetime(reviews["date"])

for k, business_id in enumerate(business_ids):
    if k % 100 == 0:
        print(f"[{k}] - Conversion for business_id: {business_id}")

    # get temporary dataframe of all reviews with given business_id and assign column 'Number' (Rating 0, ..., M_i)
    df_temp = (
        reviews[reviews["business_id"] == business_id]
        .reset_index(drop=True)
        .assign(Number=lambda x: x.index)
    )

    # calculate days since first review
    df_temp["Days"] = (df_temp["date"] - df_temp["date"].iloc[0]).dt.days
    days.extend(df_temp["Days"].tolist())
    sentiment.extend(df_temp["sentimenttext"].tolist())
    ratings.extend(df_temp["stars"].tolist())
    time.append(len(df_temp))
    age.append(df_temp["Days"].iloc[-1])


# validation checks
assert sum(time) == len(sentiment)
assert len(days) == len(sentiment)
assert len(ratings) == len(sentiment)
print("Done ... validation checks passed!")
print("Created required lists for MCMC sampling")

[0] - Conversion for business_id: QkG3KUXwqZBW18A9k1xqCA
[100] - Conversion for business_id: yTTEhnUOirRjGjxkObQTVw
[200] - Conversion for business_id: OdViVhR2ayppzkN2WtIScw
[300] - Conversion for business_id: WiqmuzPxWGiWDPxdSuVCXw
[400] - Conversion for business_id: yH83tf58E9jrME5U4HGAMg
[500] - Conversion for business_id: -PJgh1XoQBMnnSgg6MhmMA
[600] - Conversion for business_id: P7j_K9baGxWPlInbjn0OOg
[700] - Conversion for business_id: jcw_vtqfZSTLP15DJmvfIA
[800] - Conversion for business_id: OH3baEaklANPe1farAKgRg
[900] - Conversion for business_id: 3dsvREiTlmGaaBjsBS4dwQ
Done ... validation checks passed!
Created required lists for MCMC sampling


In [7]:
assert (
    not "Age" in business_covariates
), "The key Age is already added to dataframe! Make sure, that you only run this cell once!"

# add restaurant age in days to dataframe
business_covariates["Age"] = age

# change checkin count to checkin rates (number of checkins every month, assuming a month contains 28 days)
business_covariates["Checkin"] = (
    business_covariates["Checkin"] / business_covariates["Age"] * 28
)

# add log of age to dataframe for later analysis
business_covariates["logAge"] = np.log(business_covariates["Age"])

# only get relevant covariates for training
relevant_covariates = business_covariates[
    [
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Age",
    ]
].copy()

relevant_covariates

,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,American,1,2.0,700.0,350.0,1248.0,2057
1,15,0.744788,Salad,1,1.0,250.0,100.0,1638.0,2782
2,10,3.438404,American,0,1.0,247.0,186.0,1431.0,2557
3,2,1.276453,Asian,0,1.0,260.0,148.0,1223.0,2391
4,6,0.317627,Pizza,1,1.0,62.0,22.0,1398.0,2292
...,...,...,...,...,...,...,...,...,...
916,1,2.700297,American,0,3.0,207.0,128.0,1892.0,1348
917,11,25.242424,Cafes,0,2.0,496.0,356.0,1488.0,132
918,2,2.508399,Cafes,1,1.0,206.0,112.0,1288.0,893
919,2,1.698153,Cafes,0,1.0,119.0,64.0,1303.0,2869


In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

assert (
    len(relevant_covariates.columns) == 9
), "One Hot Coding and Scaling already performed on dataframe!"

# Mark as categorial variable
relevant_covariates["category"] = relevant_covariates["category"].astype("category")

# One Hot Encoding
relevant_covariates_encoded = pd.get_dummies(
    relevant_covariates, columns=["category"], drop_first=False, dtype=int
)

## Create model matrix (cov_mat)

# First two numeric columns before category column
first_numeric = ["density", "Checkin"]

# Category dummies (alphabetically sorted)
category_cols = sorted(
    [col for col in relevant_covariates_encoded.columns if col.startswith("category_")]
)

# Remaining numeric columns after category in original order
remaining_numeric = [
    "chain",
    "Price.Level",
    "Restaurant.Size",
    "Number.of.Seats",
    "ZRI",
    "Age",
]

# combine original order
column_order = first_numeric + category_cols + remaining_numeric
relevant_covariates = relevant_covariates_encoded[column_order]

# remove category_Other (8th column)
if len(relevant_covariates.columns) > 7:
    col_to_remove = relevant_covariates.columns[7]
    print(f"Removing column at index 7 (category_Other): '{col_to_remove}'")

    # Verify it's categoryOther
    if "Other" in col_to_remove:
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])
    else:
        # if not expected
        print(f"WARNING: Expected 'categoryOther' but found '{col_to_remove}'")
        print(f"All columns: {relevant_covariates.columns.tolist()}")

        # Still remove it to match R behavior
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])

print(f"Final covariate matrix shape: {relevant_covariates.shape}")
print(f"Columns: {relevant_covariates.columns.tolist()}")

relevant_covariates.head(1)

Removing column at index 7 (category_Other): 'category_Other'
Final covariate matrix shape: (921, 16)
Columns: ['density', 'Checkin', 'category_American', 'category_Asian', 'category_Cafes', 'category_Fast Food', 'category_Mexican', 'category_Pizza', 'category_Salad', 'category_Speciality Food', 'chain', 'Price.Level', 'Restaurant.Size', 'Number.of.Seats', 'ZRI', 'Age']


,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,1,0,0,0,0,0,0,0,1,2.0,700.0,350.0,1248.0,2057


In [ ]:
# First fit on training data, then center, then impute

imputer = SimpleImputer(strategy="median")  # impute column with median value
scaler = StandardScaler(with_std=False)  # only centering, no scaling!

X_train = relevant_covariates.iloc[:n_train].copy()

# First fit imputer on training data (to get medians for each column)
imputer.fit(X_train)

# Then fit scaler on imputed training data (to get means for centering)
X_train_imputed = imputer.transform(X_train)
scaler.fit(X_train_imputed)

# Now apply both transformations to all data
cov_mat_imputed = imputer.transform(relevant_covariates)
cov_mat_preprocessed = scaler.transform(cov_mat_imputed)

X_train_preprocessed = cov_mat_preprocessed[:n_train]

# QR-decomposition
Q, R = np.linalg.qr(X_train_preprocessed)

# scale the Q and R matrix appropriately
Q_scaled = Q * np.sqrt(n_train - 1)
R_scaled = R / np.sqrt(n_train - 1)

X_test = cov_mat_preprocessed[n_train:]

display(X_train)
display(pd.DataFrame(cov_mat_preprocessed))

,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,1,0,0,0,0,0,0,0,1,2.0,700.0,350.0,1248.0,2057
1,15,0.744788,0,0,0,0,0,0,1,0,1,1.0,250.0,100.0,1638.0,2782
2,10,3.438404,1,0,0,0,0,0,0,0,0,1.0,247.0,186.0,1431.0,2557
3,2,1.276453,0,1,0,0,0,0,0,0,0,1.0,260.0,148.0,1223.0,2391
4,6,0.317627,0,0,0,0,0,1,0,0,1,1.0,62.0,22.0,1398.0,2292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,11,12.242245,0,0,0,0,0,0,0,1,1,1.0,60.0,18.0,1488.0,677
496,3,0.601227,0,0,0,0,1,0,0,0,0,1.0,39.0,15.0,1515.0,2608
497,4,9.050505,1,0,0,0,0,0,0,0,0,1.0,75.0,48.0,1469.0,99
498,3,2.994907,0,0,0,0,0,1,0,0,0,2.0,135.0,40.0,1488.0,1178


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,-1.83,-3.161076,0.74,-0.092,-0.084,-0.112,-0.18,-0.104,-0.042,-0.062,0.708,0.548,470.538,251.624,-237.89,639.8
1,2.17,-4.743950,-0.26,-0.092,-0.084,-0.112,-0.18,-0.104,0.958,-0.062,0.708,-0.452,20.538,1.624,152.11,1364.8
2,-2.83,-2.050333,0.74,-0.092,-0.084,-0.112,-0.18,-0.104,-0.042,-0.062,-0.292,-0.452,17.538,87.624,-54.89,1139.8
3,-10.83,-4.212284,-0.26,0.908,-0.084,-0.112,-0.18,-0.104,-0.042,-0.062,-0.292,-0.452,30.538,49.624,-262.89,973.8
4,-6.83,-5.171111,-0.26,-0.092,-0.084,-0.112,-0.18,0.896,-0.042,-0.062,0.708,-0.452,-167.462,-76.376,-87.89,874.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,-11.83,-2.788441,0.74,-0.092,-0.084,-0.112,-0.18,-0.104,-0.042,-0.062,-0.292,1.548,-22.462,29.624,406.11,-69.2
917,-1.83,19.753687,-0.26,-0.092,0.916,-0.112,-0.18,-0.104,-0.042,-0.062,-0.292,0.548,266.538,257.624,2.11,-1285.2
918,-10.83,-2.980339,-0.26,-0.092,0.916,-0.112,-0.18,-0.104,-0.042,-0.062,0.708,-0.452,-23.462,13.624,-197.89,-524.2
919,-10.83,-3.790585,-0.26,-0.092,0.916,-0.112,-0.18,-0.104,-0.042,-0.062,-0.292,-0.452,-110.462,-34.376,-182.89,1451.8


In [ ]:
from helpers import comp_entropy

# aggregate review stats
review_stats = (
    reviews.groupby("business_id")
    .agg(
        VAR=("stars", "var"),
        MEAN=("stars", "mean"),
        ENTR=("stars", lambda x: comp_entropy(x)),
        COUNT=("stars", "size"),
        ONE_STAR=("stars", lambda x: (x == 1).sum()),
        TWO_STAR=("stars", lambda x: (x == 2).sum()),
        THREE_STAR=("stars", lambda x: (x == 3).sum()),
        FOUR_STAR=("stars", lambda x: (x == 4).sum()),
        FIVE_STAR=("stars", lambda x: (x == 5).sum()),
    )
    .reset_index()
)

# mutate count into probabilities
for col in ["ONE_STAR", "TWO_STAR", "THREE_STAR", "FOUR_STAR", "FIVE_STAR"]:
    review_stats[col] = review_stats[col] / review_stats["COUNT"]

# Add variation coeffient to review_stats
review_stats["COV"] = np.sqrt(review_stats["VAR"]) / review_stats["MEAN"]

# select covariates, that are relevant for training the benchmark models
benchmark_covariates = business_covariates[
    [
        "business_id",
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Distance.To.City.Centre",
        "Age",
        "is_open",
    ]
].copy()

# Add Closed column (opposite from is_open)
benchmark_covariates["Closed"] = 1 - benchmark_covariates["is_open"]

# merge covariates with aggregate review_stats
benchmark_covariates = benchmark_covariates.merge(
    review_stats, on="business_id", how="left"
)

# add logarithmic count
benchmark_covariates["l_COUNT"] = np.log(benchmark_covariates["COUNT"])

# Convert to categorical again
benchmark_covariates["category"] = benchmark_covariates["category"].astype("category")

# Convert Closed to categorical with proper labels (Closed = 1, Open = 0)
benchmark_covariates["Closed"] = (
    benchmark_covariates["Closed"].map({1: "Closed", 0: "Open"}).astype("category")
)

print(f"Benchmark covariates prepared with shape: {benchmark_covariates.shape}")

benchmark_covariates

/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Benchmark covariates prepared with shape: (921, 24)


,business_id,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre,...,MEAN,ENTR,COUNT,ONE_STAR,TWO_STAR,THREE_STAR,FOUR_STAR,FIVE_STAR,COV,l_COUNT
0,QkG3KUXwqZBW18A9k1xqCA,11,2.327662,American,1,2.0,700.0,350.0,1248.0,14035.314000,...,2.648649,1.421063,37,0.432432,0.108108,0.081081,0.135135,0.243243,0.643046,3.610918
1,FKNvQvsknpNSG26hVtzJ0w,15,0.744788,Salad,1,1.0,250.0,100.0,1638.0,26979.001882,...,2.315068,1.483641,73,0.356164,0.260274,0.164384,0.150685,0.068493,0.557075,4.290459
2,bVqNtwcwIz2gAacHWYF2lw,10,3.438404,American,0,1.0,247.0,186.0,1431.0,17458.501567,...,3.555556,1.509186,126,0.150794,0.087302,0.166667,0.246032,0.349206,0.401793,4.836282
3,19LQmaLOi22V5RrI3eB5lA,2,1.276453,Asian,0,1.0,260.0,148.0,1223.0,7467.113589,...,4.228571,1.170158,35,0.057143,0.028571,0.114286,0.228571,0.571429,0.269549,3.555348
4,f5hKCevF_qoC3tWkQHUS9w,6,0.317627,Pizza,1,1.0,62.0,22.0,1398.0,22542.744018,...,1.619048,0.887242,21,0.714286,0.095238,0.047619,0.142857,0.000000,0.689892,3.044522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,bBxctA7YVbZgpExmsVK7tg,1,2.700297,American,0,3.0,207.0,128.0,1892.0,11371.116772,...,4.266667,1.090333,30,0.066667,0.033333,0.033333,0.300000,0.566667,0.267819,3.401197
917,cILk7PnJBxNsMmGhQU2cyA,11,25.242424,Cafes,0,2.0,496.0,356.0,1488.0,7619.370794,...,3.592593,1.448108,27,0.111111,0.148148,0.074074,0.370370,0.296296,0.380241,3.295837
918,p_V8gH43zZx0Gh_EA__hdg,2,2.508399,Cafes,1,1.0,206.0,112.0,1288.0,14907.338168,...,4.028571,0.989680,35,0.171429,0.057143,0.000000,0.114286,0.657143,0.392415,3.555348
919,HkivRKsvPfSag81YgcLuwg,2,1.698153,Cafes,0,1.0,119.0,64.0,1303.0,8892.917199,...,4.352941,1.076217,68,0.014706,0.029412,0.088235,0.323529,0.544118,0.201469,4.219508


In [ ]:
from helpers import ModelData
from constants import PROCESSED_DATA_FOLDER

# Create ModelData instance with all data in one place
model_data = ModelData(
    n_states=None,  # not used yet, reserved for HMM models (prepare_stan_data function)
    n_total=len(time),
    n_train=n_train,
    n_obs=int(np.sum(time)),
    n_covs=cov_mat_preprocessed.shape[1],
    time=time,
    closed=1 - business_covariates["is_open"].values,
    days=days,
    ratings=ratings,
    sentiment=sentiment,
    Q=Q_scaled,
    R=R_scaled,
    X_test=X_test,
    imputer=imputer,
    scaler=scaler,
    train_indices=np.arange(500),
    calibration_indices=calibration_indices,
    eval_indices=eval_indices,
    business_covariates=business_covariates,
    cov_mat=cov_mat_preprocessed,
    benchmark_covariates=benchmark_covariates,
)

# save to pickle
output_path = PROCESSED_DATA_FOLDER / f"processed_data_{SEED}.pkl"
model_data.to_pickle(output_path)

print(f"Data saved to {output_path}")
print(model_data.summary())

Data saved to /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/processed/processed_data_42.pkl

ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 100
- Eval indices: 321

Benchmark data: Available
- Benchmark covariates shape: (921, 24)
- Columns: business_id, density, Checkin, category, chain...

